

<font color=#009afa size=50>World Bank Development Indicators</font>

<font size=50>Climate Change Analysis</font>

**Author:** Tom Rowan, ST20285213

**Course:** DAT4004_S2_23

**Course Tutor:** Dr.Priyatharshini Rajaram


<a id="0_intro"></a>
<font color=#009afa><h1>Introduction</h1></font>


>*The objective of the proposed data analysis tasks is to provide up-to-date and meaningful analysis that will help users to inform decisions on future policy and services.*

In this report, I will review a variety of data from the **World Bank Development Indicators** datasets. These include use of land, electricity generation methods and volumes of CO2 produced by source, along with population and GDP data.

Climate Change is highly likely to become more and more of a subject of interest to policy makers over the coming decades. To some, it represents and existential threat to the entire ecosystem, while others deny that it exists at all.

Lack of consensus and diverse opinion set Climate Change issues to be hard fought over in future elections globally.

This report aims to arm policy makers with useful insights.



<a id="0_contents"></a>
<font color=#00c382><h2>Report Contents</h2></font>

[Task 0. Introduction. Configure Notebook Environment](#0_intro)<br>
>[Check Internet Connectivity]<br>
[Import Libraries]<br>


[Task 1. Import and Preprocess Data Sets](#1_intro)<br>
> [Read in and process UNSD Countries and Region Dataset](#1_unsd)<br>
Mapping Regions and Subregions<br>
[Read in and process World Bank Development Indicator Dataset](#1_worldbank)<br>
Dataset processing<br>
Remove Null Values<br>
[Concatenate UNSD and World Bank datasets](#1_merge)<br>
Separate Pre-aggregated Data, and Discuss Outliers<br>

[Task 2. Data Analysis](#2_intro)<br>
> [Q1: Are CO2 emissions increasing over time?](#2_q1_intro)<br>
[Q2: Rank CO2 emitters by region. Who is worst?](#2_q2_intro)<br>
[Q3: Do richer nations generate more CO2?](#2_q3_intro)<br>
[Q4: Is there a link between population and birth/death rate to CO2 emissions?](#2_q4_intro)<br>
[Q5: Describe the Trends in Rural vs Urban land use by Region.](#2_q5_intro)<br>
[Q6: Does Land Usage Type Corollate to Total CO2 Emissions?](#2_q6_intro)<br>
[Q7: Show electrical consumption over time, by region.](#2_q7_intro)<br>
[Q8: Is there a correlation between electricity production method and overall CO2 emissions?](#2_q8_intro)<br>


[Task 3. Classification/Clustering](#3_intro)<br>
> KMeans Clustering<br>
Hierarchical Clustering<br>

[Task 4. Discussion](#4_intro)<br>
> How these findings relate to policy decisions<br>



### Check for Internet Connectivity

This notebook is saved with all code blocks run to completion, so all intended graphics should be present. A free online account is required to create the PlotLy figures, and will require live Internet access if you wish to re-run this notebook. In the event that there is no Internet connectivity, this notebook will skip re-generation any Plotly figures, rather than produce timeouts.

If the notebook detects Internet connectivity it will also pull the datasets from the Internet; else it will expect a local CSV file.

In [ ]:
# Import Standard Python Libraries
import os
import re
import json
from urllib import request
import math
import statistics
from cycler import cycler

# Coding Syntax Note: _variable_name --> temporary / locally used variable, discarded after use.

online = False
_urls_to_check = ['http://connectivitycheck.gstatic.com/generate_204', 
                  'https://github.com',
                  'https://cardiffmet.ac.uk']

def check_internet(url_list):
    # We only need the first 200 code to prove a connection.
    for _url in url_list:
        try:
            request.urlopen(_url, timeout=3)
            return True
        except request.URLError as err:
            print (f"[Check Internet] Failed for {_url}")
        
if check_internet(_urls_to_check ) == True:
    print ("[Check Internet] Internet Connectivity Detected. Using Internet resources.")
    online = True

    url_prefix = "https://datascience.daemonchild.com/year1/dat4004/"
    
else:
    print ("[Check Internet] No Internet Connectivity. Using local resources.")
    url_prefix = "../"

### Import Libraries

If you have any issues with importing the various libraries, a requirements file is included.

```
jupyter
pandas
matplotlib
scikit-learn
ipykernel
seaborn
plotly
chart_studio
nbformat
tabulate
```

In [ ]:
# Install Requirements Using Pip
#%pip install -r ../requirements.txt

In [ ]:
# Pandas and numpy
import pandas as pd
import numpy as np

# Matplot Lib
import matplotlib.pyplot as plt
from matplotlib import colormaps
from matplotlib.pyplot import figure
%matplotlib inline

# SciKit Learn
from sklearn.model_selection import train_test_split
from sklearn import linear_model
from sklearn.preprocessing import normalize
import scipy.cluster.hierarchy as shc
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import r2_score
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.stats import zscore

# Seaborn Graphs
import seaborn as sns

# Plotly
if online:
    print ("[Plotly] We're online, so importing and configuring Plotly API")
    import chart_studio
    import plotly.figure_factory as ff
    import chart_studio.plotly as py
    import plotly.graph_objects as go
    import plotly.express as px
    plotly_username = 'dg9tcm93yw4'
    plotly_key = 'yKjx7boV2HS3TcjezK5L'     # <-- Throw away account for this project
    chart_studio.tools.set_credentials_file(username=plotly_username, api_key=plotly_key)

# Tabulate
from tabulate import tabulate

# Do not print warnings!
import warnings
warnings.filterwarnings('ignore')

---
<a id="1_intro"></a>
<font color=#009afa><h1>Task 1 - Import and Preprocess Data Sets</h1></font>

In this section of this report, I will import two datasets, and combine useful data from each.

<a id="1_unsd"></a>
<font color=#00c382><h2>UNSD Countries and Region Dataset</h2></font>

The United Nations provides a simple dataset which classifies each country by region. This data will be used to enrich analysis of the main datasets.

To explore the relative size of the regions, I have performed counting operations, and plotted the results. The data is relatively clean and requires only the removal of non-relevant columns. The country names in this data set are 'better' than those in the World Bank datasets, and this data includes regions ("Americas") and subregions ("Northern Europe") that can be mapped to countries in the World Bank dataset. This will give two levels of aggregation - global regions, and subregions.

Dataset source: https://unstats.un.org/unsd/methodology/m49/overview/

Accessed: 15/05/2024

Note that it is not possible to provide a link directly to a csv file due to the javascript-based download process on this page.

I have put the dataset online here: https://datascience.daemonchild.com/year1/dat4004/datasets/unsd/unsd-countrycodes.csv

In [ ]:
# Load Data Set
_csv_url = f"{url_prefix}datasets/unsd/unsd-countrycodes.csv"
print (f"Loading data from {_csv_url}")
countries_df = pd.read_csv(_csv_url, delimiter=";")

# Rename Columns that we want to keep that have spaces in them.
countries_df.rename(columns={'Country or Area': 'Country', 'Region Name': 'Region', 'ISO-alpha3 Code': 'ISOcode','Sub-region Name': 'Subregion','Intermediate Region Name': 'Intregion'}, inplace=True)

# Fill in blanks with a zero.
countries_df = countries_df.fillna(0)
countries_df.columns


Many of the Intermediate region values are empty, and one of the subregions is also empty:

In [ ]:
print (countries_df.loc[countries_df['Intregion'] == 0].shape)
print (countries_df.loc[countries_df['Subregion'] == 0].shape)

In [ ]:
# The subregion and intermediate regions are combined, using the more localised 'Intermediate region' in preference.
countries_df['Intregion'] = np.where(countries_df['Intregion'] == 0, countries_df['Subregion'], countries_df['Intregion'])

# Remove the Subregion column and all others not required, and rename  Intregion
countries_df = countries_df.loc[:, ['Country','Region','ISOcode','Intregion']]
countries_df.rename(columns={'Intregion': 'Subregion'}, inplace=True)

print(countries_df.head())
print(countries_df.tail())


In [ ]:
print (str(countries_df.loc[countries_df['Subregion'] == 0].shape) + " <-- Antarctica!")

In [ ]:
# Manually Label Antartica for completeness; there should be no data anyway.
print (countries_df.loc[countries_df['Country'] == "Antarctica"])
countries_df.loc[countries_df['Country'] == "Antarctica", "Region"] = "Antarctica"
countries_df.loc[countries_df['Country'] == "Antarctica", "Subregion"] = "Antarctica"
print ()
print (countries_df.loc[countries_df['Country'] == "Antarctica"])

In [ ]:
# Produce Lists
countries = list(countries_df['Country'].astype(str).unique())
regions = list(countries_df['Region'].astype(str).unique())
subregions = list(countries_df['Subregion'].astype(str).unique())
country_codes = list(countries_df['ISOcode'].astype(str).unique())

# Sort these...
countries.sort()
regions.sort()
subregions.sort()
country_codes.sort()
print (f"Regions ({len(regions)}): {regions}")
print ()
print (f"Subregions ({len(subregions)}): {subregions}")
print ()
print (f"Countries ({len(countries)}): {countries[0:5]}...")
print ()
print (f"Countycodes ({len(country_codes)}): {country_codes[0:5]}...")

In [ ]:
# Show link between region and subregion
print ("Regions and Subregion members (in rows):")
subregion_gb = countries_df.groupby(by=['Region'])
print (tabulate(subregion_gb.Subregion.unique()))
print()
print ("Subregions per Region")
region_count_table = countries_df[['Country','Region']].groupby(['Region'])['Country'].count()
print (region_count_table)
print ()
print ("Countries per Subregion")
subregion_count_table = countries_df[['Country','Subregion']].groupby(['Subregion'])['Country'].count()
print (subregion_count_table)

### Define Graph Colour Maps

Different charting tools use slightly different colour map formats. In this code section, I take one colour map, and convert it into the formats required, giving all charts the same feel where possible.

In [ ]:
cmap = plt.colormaps['terrain']  
cmap_terrain_as_list = []
from pylab import *

cmap_terrain_as_list = []

for i in range(cmap.N):
    rgba = cmap(i)
    # rgb2hex accepts rgb or rgba
    cmap_terrain_as_list.append(matplotlib.colors.rgb2hex(rgba))

# For Plotly Maps
# Generate Scaled Colour Maps - Regions and Subregions
_divisor = int(len(cmap_terrain_as_list)/len(subregions))
subregion_cmap = {}
_count = 0
for _subregion in subregions:
    subregion_cmap[_subregion] = cmap_terrain_as_list[::_divisor][_count]
    _count += 1

_divisor = int(len(cmap_terrain_as_list)/len(regions))
region_cmap = {}
_count = 0
for _region in regions:
    region_cmap[_region] = cmap_terrain_as_list[::32][_count]
    _count += 1

Simple graphing of this data to show the scale of the data under various groupings.

In [ ]:
sns.set_theme()
plt.figure(figsize=(12, 7))
ax = sns.barplot(region_count_table, palette=cmap_terrain_as_list[::32], legend=False);
ax.set_yticks(ax.get_yticks())
ax.set_xticks(ax.get_xticks())
ax.set_xticklabels(regions, rotation=0, ha='center')
plt.xlabel('Region', fontsize = 15)
plt.ylabel('Number of Countries', fontsize = 15)
plt.title('Number of Countries per Region', fontsize = 20)
plt.show();

In [ ]:
sns.set_theme()
plt.figure(figsize=(15, 8))
ax = sns.barplot(subregion_count_table, palette=cmap_terrain_as_list[::32], legend=False);
ax.set_yticks(ax.get_yticks())
ax.set_xticks(ax.get_xticks())
ax.set_xticklabels(subregions, rotation=90, ha='center')
plt.xlabel('Subregion', fontsize = 15)
plt.ylabel('Number of Countries', fontsize = 15)
plt.title('Number of Countries per Subregion', fontsize = 20)
plt.show();

### Mapping these Regions



Maps are the best way to visualise the regional groupings:

In [ ]:
if online:
    fig = px.choropleth(countries_df, locations='ISOcode', color='Region', hover_name='Region',
                        projection='equirectangular', title='Region Groupings', color_discrete_map=region_cmap)
    fig.update_layout(margin={"r":0,"t":50,"l":0,"b":50}, dragmode=False)
    fig.show()
else:
    print ("[Plotly] No Internet access detected. Map cannot be created.")

In [ ]:
if online:
    fig = px.choropleth(countries_df, locations='ISOcode', color='Subregion', hover_name='Subregion',
                        projection='equirectangular', title='Sub-region Groupings', color_discrete_map=subregion_cmap)
    fig.update_layout(margin={"r":0,"t":50,"l":0,"b":50}, dragmode=False)
    fig.show()
else:
    print ("[Plotly] No Internet access detected. Map cannot be created.")

Useful variables set in the above section:

| Variable Name | Elements | Type |
|--|--|--|
| countries_df | Country,Code,Region | Pandas DF |
| countries | Country Names | List |
| regions | Region Names | List |
| subregions | Subregion Names | List |
| country_codes | Country Codes | List |
| regions_cmap | Colours  | Dict |
| subregions_cmap | Colours | Dict |

<a id="1_worldbank"></a>
<font color=#00c382><h2>World Bank Development Indicator Data</h2></font>

The World Bank Development Indicators dataset includes use of land, electricity generation methods and volumes of CO2 produced by source, along with population data. The series are all extracted into one csv file.

Reference: https://databank.worldbank.org/reports.aspx?source=2&Topic=19#advancedDownloadOptions

Accessed: 10/05/2024

As before, it is not possible to provide a direct download link due to the JavaScript based slection and download process on the website. The CSV file is available here:

https://datascience.daemonchild.com/year1/dat4004/datasets/wb/P_Data_Extract_From_World_Development_Indicators.csv

In [ ]:
# Load Data Set - Bottom 5 rows are blank or not data.
_csv_url = f"{url_prefix}datasets/wb/P_Data_Extract_From_World_Development_Indicators.csv"
print (f"Loading data from {_csv_url}")
devindic_df = pd.read_csv(_csv_url,skipfooter=5, engine='python')


In [ ]:
print (f"Dataset Column Names: {devindic_df.columns}")
print (' becomes --> ')
# Drop columns with No data
devindic_df.drop("2022 [YR2022]",axis=1,inplace=True)
devindic_df.drop("2023 [YR2023]",axis=1,inplace=True)

# Rename Columns to more useful names
numerical_fields = [str(x).zfill(2) for x in range(2000,2022)]
string_fields = ['SeriesName', 'SeriesCode', 'Country_WB', 'ISOcode']   # <-- Country_WB for WorldBank name, allows merge later
devindic_df.columns = (string_fields + numerical_fields)

print (f"Dataset Column Names: {devindic_df.columns}")
print ()
print (f"Dataset Shape: {devindic_df.shape}")
print ()
print (f"Dataset Total Size: {devindic_df.size}")
print ()


In [ ]:
# Define lists
series_codes = (list(devindic_df['SeriesCode'].astype(str).unique()))
series_names = (list(devindic_df['SeriesName'].astype(str).unique()))

### Series Code Analysis

This dataset comprises multiple subtables of unrelated data concatenated, sharing the same column names. The Series Code is the key to each sub-table, each containing a complete set of records for each country. Here I am investating the series code. It is made up of a number of tokens which identify the dataset purpose. Understanding this code will allow easier searching of the data series for relevant data.

In [ ]:

# Extract Dataset containing Series Information
series_data_df= pd.DataFrame()
series_data_df['SeriesCode'] = devindic_df['SeriesCode'].unique()
series_data_df['SeriesName'] = devindic_df['SeriesName'].unique()

series_data_df = series_data_df.sort_values(by='SeriesCode')

#print (tabulate(series_data_df, tablefmt="pipe", headers="keys"))


Creating a Unit column by splitting the "SeriesName" column - the name of the series - into two parts.

For example "Agricultural land (sq. km)" becomes SeriesName: "Agricultural Land", Unit: "SQ. KM".


In [ ]:
_temp_list = []

for _index, _row in series_data_df.iterrows():

    _code_tokens = _row['SeriesCode'].split(".")
    _num_tokens = len(_code_tokens)
    
    if _num_tokens == 3:
        _row['UnitCode'] = "--"
    elif _num_tokens == 4:
        _row['TypeCode'] = '-'.join(map(str,_code_tokens[1:3]))
        _row['UnitCode'] = _code_tokens[-1]  
    else:
        _row['UnitCode'] = _code_tokens[-1]
        _row['TypeCode'] = '-'.join(map(str,_code_tokens[1:(_num_tokens-2)]))

    _row['StudyCode']= _code_tokens[0]

    _name_tokens = _row['SeriesName'].split("(")
    if len(_name_tokens)> 1:
        _row['GraphUnit'] = _name_tokens[1].rstrip(")").upper()
    else:
        _row['GraphUnit'] = "People"
    _temp_list.append(_row)

# Finally replace into series_data_df
series_data_df = pd.DataFrame(_temp_list)


In [ ]:
print (tabulate(series_data_df, tablefmt="pipe", headers="keys"))

In [ ]:
# Merge this data into the main Development Indicators dataframe.

series_data_df.to_csv("../outputs/wb_series_codes_analysis.csv")

# Drop Series Name from one table
series_data_df.drop(['SeriesName'], axis=1, inplace=True)

# Back fill from the series analysis above

_columns_to_merge = series_data_df.columns.difference(devindic_df.columns)
devindic_df = pd.merge(devindic_df, series_data_df, on=["SeriesCode"], how='left')

devindic_df.columns


In [ ]:
devindic_df.tail(2)

### Null Values

The dataset contains a large number of elements where there is a null value. This has been recorded in the dataset as a '..' value. For the purposes of this study, unreported figures and zero figures are equivalent, as a zero figure is in most cases highly unlikely, excepting perhaps small island nations.

In [ ]:
# Showing a line in the dataset with multiple '..' values:
devindic_df.loc[devindic_df['Country_WB'] == "Sudan"].loc[devindic_df['SeriesCode'] == "AG.LND.EL5M.UR.K2"]

In [ ]:
# Replacing the ".." marker in all numerical data columns.
for column in numerical_fields:
    devindic_df[column] = devindic_df[column].replace("..","0")

In [ ]:
# Showing the same line in the dataset:
devindic_df.loc[devindic_df['Country_WB'] == "Sudan"].loc[devindic_df['SeriesCode'] == "AG.LND.EL5M.UR.K2"]

<a id="1_merge"></a>
<font color=#00c382><h2>Merge UNSD and WorldBank Tables</h2></font>

I am merging the tables based on the ISOcode for the country.

In [ ]:
country_data_df = pd.merge( devindic_df, countries_df, on=["ISOcode"], how='left')

# Remove the World Bank Country Name (they are not as readable as the UN country names)
country_data_df.drop(columns=['Country_WB'], inplace=True)

Set Column Types now that we have built the entire dataframe.

In [ ]:

# Generate a list of the non-numerical column names
print ("Non-numerical fields:")
print (f"{string_fields}")
print (' becomes --> ')
string_fields = [sf for sf in country_data_df.columns if sf not in numerical_fields]
print (string_fields)

for column in string_fields:
   country_data_df[column] = country_data_df[column].astype(str)

for column in numerical_fields:
   country_data_df[column] = country_data_df[column].astype(float)

country_data_df.dtypes

# Write out to CSV
country_data_df.to_csv("../outputs/wb_merged_unsd_on_isocode.csv", sep=',', encoding='utf-8')

# Save memory by deleting source datasets
del devindic_df
del countries_df

### Separate Aggregated Data, and Discuss Outliers
As well as data for distinct countries, the World Bank dataset also includes data from their own aggregated regions. This data can be extracted from the working data set and saved. The Kosovo country code is not found in the UNSD data. It is an empty row and can also be removed, as shown from this extract from the CSV:

> Agricultural Land,AG.LND.AGRI.K2,Kosovo,XKX,..,..,..,..,..,..,..,..,..,..,..,..,..,..,..,..,..,..,..,..,..,..,..,..

To do this, I am using the country_codes extracted from the UNSD dataset, and using the intersection between the "ISOcode" column to filter the data into two dataframes. I have saved these to CSV once completed.

In terms of outliers, there is a huge variation in the values within the dataset. Consider the area of 'China' vs 'Jersey', and yet both extremes are equally valid for this study. Only countries with no data need be removed - and this will be best carried out once individual data series are extracted from the main data set, ready for comparison together. At that point, I will preprocess the data to remove problematic rows from the sample.

In [ ]:
# Extract Regional Data (NB if NOT in)
regional_data_only_df = pd.DataFrame(country_data_df[~country_data_df['ISOcode'].isin(country_codes)])
#regional_data_only_df[["ISOcode","Country"]].drop_duplicates(inplace=True)
regional_data_only_df.to_csv("../outputs/wb_regional_data_discarded.csv")

# Save memory
del regional_data_only_df

# Country Data
country_data_df = pd.DataFrame(country_data_df[country_data_df['ISOcode'].isin(country_codes)])
country_data_df.to_csv("../outputs/country_data.csv")


In [ ]:
print (f"Dataset Column Names: {country_data_df.columns}")
print ()
print (f"Dataset Shape: {country_data_df.shape}")
print ()
print (f"Dataset Total Size: {country_data_df.size}")
print ()

In [ ]:
#To extract an individual sub table, the following example can be used:
    
df_AG_LND_AGRI_K2 = country_data_df.loc[country_data_df['SeriesCode'] == "AG.LND.AGRI.K2"]
df_AG_LND_AGRI_K2.head(3)

I have define a simple graphing function to allow me to loop through the datasets and produce a time series graph to view the shape and magnitude of the data.

In [ ]:
def plot_data_series_years (dataframe, x_column='Country', ylabel="y axis", title="a graph", start=2000, end=2021, log=0, showlegend=False):

    _range = [str(x).zfill(2) for x in range(start,end+1)]
    _years = pd.Index(_range)

    for _index, _row in dataframe.iterrows():
        _country = _row[x_column]
        _country_data = _row[_years]

        # There *will* likely be divide by zero errors, trying to trap, but ensure that we fill with a true zero.
        if log == 10:
            _country_data = np.log10(_country_data.astype(float), out=np.zeros_like(_country_data), where=(_country_data>0.0))
        elif log == 2:
            _country_data = np.log2(_country_data.astype(float), out=np.zeros_like(_country_data), where=(_country_data>0.0))
        
        plt.plot(_years,_country_data,label=_country);
    plt.xlabel("Year")
    plt.ylabel(ylabel)
    plt.title(title)
    fig = plt.gcf()
    fig.set_size_inches(12, 5, forward=True)

    # Autoscale colours
    _length = dataframe.shape[0]
    if _length > 0:
        _colour_divisor = int(len(cmap_terrain_as_list)/_length)+1   # <-- +1 factor mixes colour repeats
    else:
        _colour_divisor = 1

    plt.rcParams['axes.prop_cycle'] = cycler(color=cmap_terrain_as_list[::_colour_divisor])
    if showlegend:
        plt.legend(loc='center left', bbox_to_anchor=(1.0, 0.5))

    plt.show();

Using the function, I will now plot all the data series in turn to ascertain the shape of the dataset graphically.

In [ ]:
# NB run time approx 50 sec

_draw_them = True

def get_unit (dataframe):
    return str(dataframe["GraphUnit"].unique()[0])

def get_seriesname (dataframe):
    return str(dataframe["SeriesName"].unique()[0])

if _draw_them:
    for _series in series_codes:
        _series_data = country_data_df.loc[country_data_df['SeriesCode'] == _series]
        _unit = get_unit(_series_data)
        _series_name = get_seriesname (_series_data)
        _title = str(_series_name)+", by Country ("+_series+")"
        plot_data_series_years (dataframe=_series_data, x_column= 'Country', title=_title, ylabel=_unit)

This is clearly an raw and unsuitable plot for clearly showing this data, (even using logarithmic values). However, these plots how the general 'shape' of the data sets, including where holes exist in the data.

Variables set in the above section:

| Variable Name | Elements | Type |
|--|--|--|
| country_data_df| Country,Code,SeriesCode,{Data Values} | Pandas DF |
| series_code | Series Code | List |
| series_names | Series Names | List |


---
<a id="2_intro"></a>
<font color=#009afa><h1>Task 2 - Data Analysis</h1></font>



## Task Questions

The datasets available from the World Bank are extensive. I have collected together a wide selection of series from the Development Indicators range. This allows me to pose a number of questions that should provide insight for policy makers.

- Q1: Are CO2 emissions increasing over time?
- Q2: Rank CO2 emitters by region. Who is worst?
- Q3: Do richer nations generate more CO2?
- Q4: Is there a link between population and birth/death rate to CO2 emissions?
- Q5: Describe the Trends in Rural vs Urban land use by Region.
- Q6: Does Land Usage Type Corollate to Total CO2 Emissions?
- Q7: Show electrical consumption over time, by region.
- Q8: Is there a correlation between electricity production method and overall CO2 emissions?

---
<a id="2_q1_intro"></a>
<font color=#00c382><h2>Q1: Are CO2 emissions increasing over time?</h2></font>

This is an interesting question because while it is common understanding that global emissions are increasing in general, this may not be true for all regions, or countries. This dataset is limited to 2000-2020, so there will be an increase at the end due to the global Pandemic.

In this question, I will utilise data on total CO2 emissions.

I have selected the following series to answer this question:

StudyCodes: EN.ATM

|Series Code|Description|Unit|
|-|-|-|
EN.ATM.CO2E.KT | CO2 Emissions | Tons x 10^3 (KT)

In [ ]:
# Extract C02 data for all countries
co2data_df = country_data_df.loc[country_data_df['SeriesCode'] == 'EN.ATM.CO2E.KT'].dropna()

# Remove 2021 - it's empty for all countries in this dataset
co2data_df.drop('2021', axis=1, inplace=True)
co2_data_num_fields = numerical_fields = [str(x).zfill(2) for x in range(2000,2021)]


As an initial review, I have mapped the data at the extremes of the data range - 2000 and 2020 - to review changes over the time series. From these maps, it's clear to see two facts:

- Global Emissions have increased. The scale of the legend on the maps demonstrates this clearly.
- The leader board of emitters has changed over time. 

In [ ]:

fig = px.choropleth(co2data_df, locations='ISOcode', color='2000', hover_name='Country',
                        projection='equirectangular', title='CO2 Emissions by Country 2000', hover_data='2000')
fig.update_layout(margin={"r":0,"t":50,"l":0,"b":50}, dragmode=False)
fig.show()


In [ ]:
if online:
    fig = px.choropleth(co2data_df, locations='ISOcode', color='2020', hover_name='Country',
    projection='equirectangular', title='CO2 Emissions by Country 2020', hover_data='2020')
    fig.update_layout(margin={"r":0,"t":50,"l":0,"b":50}, dragmode=False)
    fig.show()
else:
    print ("[Plotly] No Internet access detected. Map cannot be created.")

Summed totals for each subregion, plotted over time:

In [ ]:
#Extract Data
co2data_subregion_gb = co2data_df.groupby(by='Subregion')[co2_data_num_fields].agg(['sum'])

# Create plot
fig, ax = plt.subplots(figsize=(15,6))
co2data_subregion_gb.transpose().plot(kind='line', ax=ax, ylabel="Kilotonnes", title="Total CO2 Emissions by Subregion");
plt.legend(loc='center left', bbox_to_anchor=(1.0, 0.5))
plt.rcParams['axes.prop_cycle'] = cycler(color=cmap_terrain_as_list[::32])
plt.show()

To show a total global output, I have applied a second summation:

In [ ]:
# Total CO2 emissions over time:
fig, ax = plt.subplots(figsize=(15,6))
co2data_subregion_gb_all = co2data_df.groupby(by='Subregion')[co2_data_num_fields].sum().apply(['sum'])
co2data_subregion_gb_all.transpose().plot(kind='line', ax=ax, ylabel="Kilotonnes",title="Global Total Emissions",legend=False);
plt.rcParams['axes.prop_cycle'] = cycler(color=cmap_terrain_as_list[::32])
plt.show();

Is it all bad news however? Who reduced their emissions by the end of the 20 year period?

In [ ]:
co2data_df['20YearDiff'] = co2data_df['2020'] - co2data_df['2000']

#co2_country_better_df = co2data_df[['Country','2000','2020']][(co2data_df['2000'] > co2data_df['2020'])]
co2_country_better_df = co2data_df[['Country','20YearDiff']][co2data_df['20YearDiff'] < 0.0]

# Rank by best reduction being top
co2_country_better_df['rank'] = co2_country_better_df['20YearDiff'].rank(ascending=True)
print (co2_country_better_df.sort_values(by='rank'))

### Conclusion

From this analysis, it is clear that emissions are increasing, globally, despite some regions and individual countries changing for the better. Generally the worst polluters are still getting worse.

Global emissions peaked in 2018. Understandably, the effect of the SARS-CoV-2 pandemic is shown as a marked decrease, but the trends was already slightly downwards. The limitations of the dataset show here.

---
<a id="2_q2_intro"></a>
<font color=#00c382><h2>Q2: Rank CO2 emitters by region. Who is worst?</h2></font>

Following on from the last analysis, I will produce a global ranking table for each subregion over time.

I will first group the data by subregion, then sum the total emissions for each subregion. Once ordered by this sum, the rank can be assigned, in decending order (where highest value = rank 1, next highest = 2).

I have selected the following series to answer this question:

StudyCodes: EN.ATM

|Series Code|Description|Unit|
|-|-|-|
EN.ATM.CO2E.KT | CO2 Emissions | Tons x 10^3 (KT)

From the information in the plots produced in Task 1, I will compare the values on a line graph using a logarithmic scale. I will sum and then perform a $\log_{10}$ operation to allow for comparison between regions. Using this plot, we can see that there is a lot of movement over twenty years, some increasing and some decreasing their output.

In [ ]:
# Extract C02 data for all countries
co2data_df = country_data_df.loc[country_data_df['SeriesCode'] == 'EN.ATM.CO2E.KT'].dropna()

# Remove 2021 - it's empty for all countries in this dataset
co2data_df.drop('2021', axis=1, inplace=True)
co2_data_num_fields = numerical_fields = [str(x).zfill(2) for x in range(2000,2021)]

#Extract Data
co2data_subregion_gb = co2data_df.groupby(by='Subregion')[co2_data_num_fields].agg(['sum'])

In [ ]:
fig, ax = plt.subplots(figsize=(15,6))
co2data_subregion_gb_log = co2data_df.groupby(by='Subregion')[co2_data_num_fields].sum().apply(lambda x: np.log10(x))
co2data_subregion_gb_log.transpose().plot(kind='line', ax=ax, ylabel="Kilotonnes",title="Total CO2 Emissions by Subregion (Log10)");
plt.legend(loc='center left', bbox_to_anchor=(1.0, 0.5))
plt.rcParams['axes.prop_cycle'] = cycler(color=cmap_terrain_as_list[::32])
plt.show();


Extract Mean value for CO2 by Region, to gain insight into whether a subregion is the highest producer, on average. This takes into account that some regions may include high volume emitters *and* at the same time include very low level emitters. This shows that Northern America has been consistenly a higher producer - on average - for longer than Eastern Asia.

In [ ]:
fig, ax = plt.subplots(figsize=(15,6))
co2data_subregion_gb_mean = co2data_df.groupby(by='Subregion')[co2_data_num_fields].agg(['mean'])
co2data_subregion_gb_mean.transpose().plot(kind='line', ax=ax, ylabel="Kilotonnes", title="Total CO2 Emissions by Subregion (Mean)");
plt.legend(loc='center left', bbox_to_anchor=(1.0, 0.5))
plt.rcParams['axes.prop_cycle'] = cycler(color=cmap_terrain_as_list[::32])
plt.show()

Plotting this as a barchart again, clearly shows the ranking:

In [ ]:
numerical_fields = [str(x).zfill(2) for x in range(2000,2021)]
sns.set_theme()
plt.figure(figsize=(12,6))
ax = sns.barplot(co2data_subregion_gb.transpose(), legend=False);
ax.set_yticks(ax.get_yticks())
ax.set_xticks(ax.get_xticks())
ax.set_xticklabels(co2data_subregion_gb.transpose().columns, rotation=90, ha='center')

plt.xlabel('Region', fontsize = 15)
plt.ylabel('Kilotonnes', fontsize = 15)
plt.title('Average CO2 Emissions, per Subregion', fontsize = 20)
plt.show();


A heatmap is another way of clearly showing the worst polluters, over time:

In [ ]:
sns.set_theme()
plt.figure(figsize=(10, 8))
ax = sns.heatmap(co2data_subregion_gb.transpose(),cmap="terrain");
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='left')
ax.set_yticklabels(co2_data_num_fields, rotation=0)
plt.xlabel('Subregions', fontsize = 15)
plt.ylabel('Years', fontsize = 15)
plt.title('Heatmap of Subregional CO2 Emission Totals', fontsize = 20)
ax.xaxis.tick_top() 


### Produce Ranking Table

To create a clear ranking table and plot, I will use a groupby with a sum, and then ranking.

In [ ]:
# Build empty DataFrame, using appropriate year fields list
co2_subregion_rank_df = pd.DataFrame(columns=co2_data_num_fields)

# Build a ranking for each year in turn, and add to th eoutput dataframe
for _year in co2_data_num_fields:
    _new_df = co2data_df.groupby(by='Subregion')[_year].agg(['sum']).sort_values(by='sum', ascending=False).rank(ascending=False)
    co2_subregion_rank_df[_year] = _new_df

print(co2_subregion_rank_df)


In [ ]:
sns.set_theme()
plt.figure(figsize=(15,7))
ax = sns.lineplot(co2_subregion_rank_df.transpose(),palette="terrain_r", linewidth = 2.2, linestyle='dashed');
ax.set_xticks(['2000','2005','2010','2015','2020'])
ax.set_yticks([25,20,15,10,5,0])
ax.set_xticklabels(ax.get_xticklabels(), rotation=0, ha='center')
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
plt.xlabel('Year', fontsize = 15)
plt.ylabel('Ranking', fontsize = 15)
plt.title('CO2 Emitters Leaderboard (2000-2020)', fontsize = 18)
ax.xaxis.tick_bottom()
ax.invert_yaxis()
plt.legend(loc='center left', bbox_to_anchor=(1.0, 0.5)) 
plt.show()

In [ ]:
sns.set_theme()
plt.figure(figsize=(12, 10))
ax = sns.heatmap(co2_subregion_rank_df,cmap="terrain_r",annot=True);
ax.set_xticklabels(co2_data_num_fields, rotation=45, ha='left')
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
ax.xaxis.tick_bottom()
plt.xlabel('Year', fontsize = 15)
plt.ylabel('Subregion', fontsize = 15)
plt.title('Heatmap of Subregional CO2 Emission Leaderboard', fontsize = 18)
plt.show()

It is also easy to produce a country rank table, although this will be limited to the Top 20 worst polluters for brevity. This limit will create holes in the data where countries drop out of this ranking.

In [ ]:
_number_to_show = 20

# Build empty DataFrame, using appropriate year fields list
co2_country_rank_df = pd.DataFrame(columns=co2_data_num_fields)

# Build a ranking for each year in turn, and add to the output dataframe
for _year in co2_data_num_fields:
    _new_df = co2data_df.groupby(by='Country')[_year].agg(['sum']).sort_values(by='sum', ascending=False).rank(ascending=False).head(_number_to_show)
    co2_country_rank_df[_year] = _new_df

print(co2_country_rank_df)

In [ ]:
sns.set_theme()
plt.figure(figsize=(15,7))
ax = sns.lineplot(co2_country_rank_df.transpose(),palette="terrain_r", linewidth = 2.2, linestyle='dashed');
ax.set_xticks(['2000','2005','2010','2015','2020'])
ax.set_yticks([25,20,15,10,5,0])
ax.set_xticklabels(ax.get_xticklabels(), rotation=0, ha='center')
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
plt.xlabel('Year', fontsize = 15)
plt.ylabel('Ranking', fontsize = 15)
plt.title('CO2 Emitters Leaderboard (2000-2020)', fontsize = 18)
ax.xaxis.tick_bottom()
ax.invert_yaxis()
plt.legend(loc='center left', bbox_to_anchor=(1.0, 0.5)) 
plt.show()

In [ ]:
sns.set_theme()
plt.figure(figsize=(12, 10))
ax = sns.heatmap(co2_country_rank_df,cmap="terrain_r",annot=True);
ax.set_xticklabels(co2_data_num_fields, rotation=45, ha='left')
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
ax.xaxis.tick_bottom()
plt.xlabel('Year', fontsize = 15)
plt.ylabel('Country', fontsize = 15)
plt.title('Heatmap of Top 20 Ranked CO2 Emitters', fontsize = 17)
plt.show()

### Conclusion

There are clear consistent winners on the CO2 Emissions Leaderboard. While the general trend is upwards, as question one shows, the rankings change. Why countries are moving in the emissions ranking is not clear without further analysis. Does the UK move from 7th to 18th because it has drastically reduced emissions, or because others have increased theirs equally drastically? What is the key driver behind these changes? Thsi report may offer some answers below.

Policy makers can use this chart to see how their country - or region - is ranked against others. This could be used to provide evidence that a policy is working (or not.)


---
<a id="2_q3_intro"></a>
<font color=#00c382><h2>Q3: Do richer nations generate more CO2?</h2></font>

In this question I will explore the relationship between the richer nations and their CO2 emissions footprint. I will use GDP per capita and CO2 emissions per capita. 

I will rank the countries by their GDP/PC and CP2/PC figures over the timeframe of the study to show variation in the 'leaderboard'. This should bring more interesting insights into other nations other than the usual 'large players'.

StudyCodes: EN.ATM

|Series Code|Description|Unit|
|-|-|-|
EN.ATM.CO2E.KT | CO2 Emissions | Tons x 10^3 (KT)
EN.ATM.CO2E.KT | GDP | Tons x 10^3 (KT)
EN.ATM.CO2E.KT | Population | Tons x 10^3 (KT)

In [ ]:
# Extract Study for all countries into a dictionary.
_source_dfs  = {}
_series_map = {
    'co2':     'EN.ATM.CO2E.KT',
    'pop':     'SP.POP.TOTL',
    'gdp':     'NY.GDP.MKTP.CD'
}

series_list = _series_map.keys()

_drop_fields = ['2021']                 
num_fields = [str(x).zfill(2) for x in range(2000,2021)]   

for _series in series_list:
    _source_dfs[_series] = country_data_df.loc[country_data_df['SeriesCode'] == _series_map[_series]]
    _source_dfs[_series].drop(columns=_drop_fields, axis=1, inplace=True)

_source_dfs.keys()

Calculate GDP / Capita and CO2 / Capita figures

In [ ]:
for _series in _source_dfs.keys():
    _source_dfs[_series].reset_index(inplace=True)
_series_list = list(_source_dfs.keys())
_series_list.remove('pop')

In [ ]:
gdp_co2_dfs = {}

for _series in ['co2', 'gdp']:
    _new_df = _source_dfs[_series][num_fields] / _source_dfs['pop'][num_fields] 
    gdp_co2_dfs[_series+'_pc'] = pd.merge(_new_df, _source_dfs[_series][['Country', 'Subregion', 'Region','ISOcode']],left_index=True, right_index=True,how='outer')

gdp_co2_dfs.keys()

In [ ]:
gdp_co2_dfs['gdp_co2_series'] = pd.DataFrame()

# Loop through datasets, extracting into one long series
for _key in ['co2_pc', 'gdp_pc']:
    _loop_df = gdp_co2_dfs[_key][num_fields]
    _df = pd.concat([_loop_df[col] for col in _loop_df])

    gdp_co2_dfs['gdp_co2_series'][_key] = list ( _df)
    #gdp_co2_dfs['gdp_co2_series'][_key+'_log10'] = list ( np.log10(_df))


gdp_co2_dfs.keys()

#del _source_dfs

In [ ]:
gdp_co2_dfs['gdp_co2_series'].loc[(gdp_co2_dfs['gdp_co2_series']!=0).all(axis=1)]

In [ ]:
gdp_co2_dfs['gdp_co2_series']['co2_pc']

In [ ]:
gdp_co2_dfs['gdp_co2_series'] = gdp_co2_dfs['gdp_co2_series'].loc[(gdp_co2_dfs['gdp_co2_series'][['co2_pc', 'gdp_pc']]!=0).all(axis=1)]

In [ ]:
gdp_co2_dfs['gdp_co2_series']['co2_pc_scaled'] = gdp_co2_dfs['gdp_co2_series']['co2_pc'].apply(lambda x: np.log10(x))
gdp_co2_dfs['gdp_co2_series']['gdp_pc_scaled'] = gdp_co2_dfs['gdp_co2_series']['gdp_pc'].apply(lambda x: np.log10(x))

### Training the Linear Regression Model

I have created a modified function to perform a linear regression and then plot the result on the same graph. The regression object is returned for future use.

In [ ]:
def scatter_plot (x_series, y_series, x_label, y_label, figsize=(10, 8), lr = "", x_train = ""):

    # Plot as a scatter plot
    plt.figure(figsize=(10, 8))
    plt.scatter(x_series, y_series, s=5)
    plt.ylabel(x_label)
    plt.xlabel(y_label)
    plt.title(f'Scatter Plot: {x_label} vs {y_label}')

    if lr:
        plt.plot(x_train,lr.coef_[0][0]*x_train + lr.intercept_[0],'-g', )
    plt.rcParams['axes.prop_cycle'] = cycler(color=cmap_terrain_as_list[::32])
    plt.show();


def linregr_and_plot (x_series, y_series, x_label, y_label):

    lin_x = pd.DataFrame(x_series)
    lin_y = pd.DataFrame(y_series)

    # Setting random_state tp ensure consistent results across notebook runs.
    x_train, x_test, y_train, y_test = train_test_split(lin_x , lin_y, test_size=0.33, random_state=69)

    regr=linear_model.LinearRegression()
    regr.fit(x_train,y_train)

    test_y_ = regr.predict(x_test)

    # Modified from workshop on Linear Regression
    print('Coefficients: ', regr.coef_)
    print('Intercept: ', regr.intercept_)
    print("Mean absolute error: %.2f" % np.mean(np.absolute(test_y_-y_test)))
    print("Residual sum of squares (MSE): %.2f" % np.mean((test_y_-y_test)**2))
    print("R2-score: %.2f" % r2_score(test_y_,y_test))

    scatter_plot (x_series, y_series, x_label, y_label, lr=regr, x_train=x_train)
    
    return regr

In [ ]:
_regr = linregr_and_plot (gdp_co2_dfs['gdp_co2_series']['co2_pc_scaled'], gdp_co2_dfs['gdp_co2_series']['gdp_pc_scaled'], "GDP", "CO2")

Given the R2 value, there is a reasonably defined relationship between high GDP per capita and CO2 emissions.

### Display the Ranking Table over Time



In [ ]:


# Build empty DataFrame, using appropriate year fields list
_co2_rank_gb_df = pd.DataFrame(columns=num_fields)

# Build a ranking for each year in turn, and add to the output dataframe
for _year in num_fields:
    _new_df = gdp_co2_dfs['co2_pc'].groupby(by='Country')[_year].agg(['sum']).sort_values(by='sum', ascending=False).rank(ascending=False)
    _co2_rank_gb_df[_year] = _new_df

_gdp_rank_gb_df = pd.DataFrame(columns=num_fields)

# Build a ranking for each year in turn, and add to the output dataframe
for _year in num_fields:
    _new_df = gdp_co2_dfs['gdp_pc'].groupby(by='Country')[_year].agg(['sum']).sort_values(by='sum', ascending=False).rank(ascending=False)
    _gdp_rank_gb_df[_year] = _new_df


gdp_co2_dfs['co2_pc_ranked'] = _co2_rank_gb_df
gdp_co2_dfs['gdp_pc_ranked'] = _gdp_rank_gb_df

del _co2_rank_gb_df
del _gdp_rank_gb_df


### CO2 per Capita Leaderboard

In [ ]:
def draw_leaderboard (dataframe, title, sort_year, top_n=20):
    
    _plot_df = dataframe.sort_values(by=sort_year).head(top_n).transpose()

    sns.set_theme()
    plt.figure(figsize=(15,7))
    ax = sns.lineplot(_plot_df, palette="terrain_r", linewidth = 2.2, linestyle='dashed');
    ax.set_xticks(ax.get_xticks())
    ax.set_yticks(ax.get_yticks())
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0, ha='center')
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
    plt.xlabel('Year', fontsize = 15)
    plt.ylabel('Ranking', fontsize = 15)
    plt.title(title, fontsize = 18)
    ax.xaxis.tick_bottom()
    ax.invert_yaxis()
    plt.legend(loc='center left', bbox_to_anchor=(1.0, 0.5)) 
    plt.show()

In [ ]:
_number_to_show = 20
draw_leaderboard (gdp_co2_dfs['co2_pc_ranked'], title='CO2 Emissions/pc Leaderboard (2000-2020) - Top '+str(_number_to_show)+' based on Year 2000 Rank', sort_year='2000')
draw_leaderboard (gdp_co2_dfs['co2_pc_ranked'], title='CO2 Emissions/pc Leaderboard (2000-2020) - Top '+str(_number_to_show)+' based on Year 2020 Rank', sort_year='2020')


In [ ]:
gdp_co2_dfs['co2_pc_ranked'].sort_values(by='2000').head(20).transpose()

A heatmap is an effective way to show these values:

In [ ]:
_number_to_show = 20

sns.set_theme()
plt.figure(figsize=(12, 10))
ax = sns.heatmap(gdp_co2_dfs['co2_pc_ranked'].sort_values(by='2000').head(_number_to_show ), cmap="terrain_r",annot=True);
ax.set_xticklabels(co2_data_num_fields, rotation=45, ha='left')
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
ax.xaxis.tick_bottom()
plt.xlabel('Year', fontsize = 15)
plt.ylabel('Country', fontsize = 15)
plt.title('CO2 Emissions/pc Leaderboard (2000-2020) - Top '+str(_number_to_show)+' based on Year 2020 Rank', fontsize = 17)
plt.show()

In [ ]:
sns.set_theme()
plt.figure(figsize=(12, 10))
ax = sns.heatmap(gdp_co2_dfs['co2_pc_ranked'].sort_values(by='2020').head(_number_to_show ), cmap="terrain_r",annot=True);
ax.set_xticklabels(co2_data_num_fields, rotation=45, ha='left')
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
ax.xaxis.tick_bottom()
plt.xlabel('Year', fontsize = 15)
plt.ylabel('Country', fontsize = 15)
plt.title('CO2 Emissions/pc Leaderboard (2000-2020) - Top '+str(_number_to_show)+' based on Year 2020 Rank', fontsize = 17)
plt.show()

### GDP per Capita Leaderboard

In [ ]:
draw_leaderboard (gdp_co2_dfs['gdp_pc_ranked'], title='CO2 Emissions/pc Leaderboard (2000-2020) - Top '+str(_number_to_show)+' based on Year 2000 Rank', sort_year='2000')
draw_leaderboard (gdp_co2_dfs['gdp_pc_ranked'], title='CO2 Emissions/pc Leaderboard (2000-2020) - Top '+str(_number_to_show)+' based on Year 2020 Rank', sort_year='2020')

Removing the Cayman Islands from the dataset to view the chart without this interesting, but anomolous country, which only started reporting figures in 2016:

In [ ]:
gdp_co2_dfs['gdp_pc'].loc[gdp_co2_dfs['gdp_pc']['Country'] == 'Cayman Islands']


In [ ]:
_temp_df = gdp_co2_dfs['gdp_pc_ranked'].drop('Cayman Islands')
draw_leaderboard (_temp_df, title='GDP/pc Leaderboard (2000-2020) - Top '+str(_number_to_show)+' based on Year 2020 Rank', sort_year='2020', top_n=20)

In [ ]:
gdp_co2_dfs['gdp_pc_ranked'].sort_values(by='2000').head(20).transpose()

In [ ]:
_number_to_show = 20

sns.set_theme()
plt.figure(figsize=(12, 10))
ax = sns.heatmap(gdp_co2_dfs['gdp_pc_ranked'].sort_values(by='2000').head(_number_to_show ), cmap="terrain_r",annot=True);
ax.set_xticklabels(co2_data_num_fields, rotation=45, ha='left')
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
ax.xaxis.tick_bottom()
plt.xlabel('Year', fontsize = 15)
plt.ylabel('Country', fontsize = 15)
plt.title('GDP Emissions/pc Leaderboard (2000-2020) - Top '+str(_number_to_show)+' based on Year 2000 Rank', fontsize = 17)
plt.show()

In [ ]:
_temp_df = gdp_co2_dfs['gdp_pc_ranked'].drop('Cayman Islands').sort_values(by='2020').head(_number_to_show)

sns.set_theme()
plt.figure(figsize=(12, 10))
ax = sns.heatmap(_temp_df, cmap="terrain_r",annot=True);
ax.set_xticklabels(co2_data_num_fields, rotation=45, ha='left')
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
ax.xaxis.tick_bottom()
plt.xlabel('Year', fontsize = 15)
plt.ylabel('Country', fontsize = 15)
plt.title('GDP/pc Leaderboard (2000-2020) - Top '+str(_number_to_show)+' based on Year 2020 Rank', fontsize = 17)
plt.show()

### Conclusion

The comparison between GDP and CO2 per capita figures reveals a different view of the data. It is possible view the data without the undue influence of the largest countries, and new insights are gained.

**CO2 emissions per Capita ranking**

There are some clear winners on the polluters leaderboard. The majority of the Top 10 are oil producers.


**GDP per Capita ranking**

The ranking of these countries by GDP/pc shows which countries are 'the richest'. This is a crude measure, as there is no accounting for the spread of wealth. However, the top three on the leaderboard were not a surprise. 

It is interesting to watch the rise of nations such as Greenland, which has moved from ranking 35th in 2000 to 15th in 2020. The extremely low population of Greenland compared to other physically expansive nations has an effect, regardless of scaling the data.

---
<a id="2_q4_intro"></a>
<font color=#00c382><h2>Q4: Is there a link between population and birth/death rate to CO2 emissions?</h2></font>



I have selected the following series to answer this question:

StudyCodes: EN.ATM

|Series Code|Description|Unit|
|-|-|-|
EN.ATM.CO2E.KT | CO2 Emissions | Tons x 10^3 (KT)
SP.POP.TOTL | Total Polulation | Value
SP.DYN.CBRT.IN | Birth rate, crude (per 1,000 people) | Value

In [ ]:
# Extract Study for all countries into a dictionary.
_source_dfs  = {}
_series_map = {
    'co2':     'EN.ATM.CO2E.KT',
    'pop':     'SP.POP.TOTL',
    'br':      'SP.DYN.CBRT.IN'
}

series_list = _series_map.keys()

_drop_fields = ['2021']                 
num_fields = [str(x).zfill(2) for x in range(2000,2021)]   

for _series in series_list:
    _source_dfs [_series] = country_data_df.loc[country_data_df['SeriesCode'] == _series_map[_series]]
    _source_dfs [_series].drop(columns=_drop_fields, axis=1, inplace=True)

_source_dfs.keys()

In [ ]:

_series_list = list(_source_dfs.keys())
_series_list .remove('co2')

# Extract Co2 values into a long series
_loop_series = _source_dfs['co2'][num_fields]
_co2_series = pd.concat([_loop_series[col] for col in _loop_series])

# Empty dictionary to store the data
co2_vs_pop_dfs = {}

# Loop through datasets, extracting into one long series
for _key in list(_series_list):

    _loop_series = _source_dfs[_key][num_fields]
    _col_list = ['co2', _key]
    co2_vs_pop_dfs[_key] = pd.DataFrame(columns = _col_list)

    _df = pd.concat([_loop_series[col] for col in _loop_series])
    co2_vs_pop_dfs[_key]['co2'] = list ( _co2_series )
    co2_vs_pop_dfs[_key][_key] = list ( _df )

co2_vs_pop_dfs['br']

del _source_dfs

In [ ]:
def log10_to_column (df, series):
    _new_series = series + "_log10"
    df[_new_series] = df[series].apply(lambda x: np.log10(x))
    return df


# Loop through the datasets, remove zero value rows and perform log10
for _key in list(co2_vs_pop_dfs.keys()):
    co2_vs_pop_dfs[_key] = co2_vs_pop_dfs[_key].loc[(co2_vs_pop_dfs[_key] > 0.0).all(axis=1)]
    log10_to_column (co2_vs_pop_dfs[_key], _key)
    log10_to_column (co2_vs_pop_dfs[_key], 'co2')


In [ ]:
print(co2_vs_pop_dfs['br'])
print(co2_vs_pop_dfs['pop'])

### Population vs CO2 Emissions

In [ ]:
scatter_plot (co2_vs_pop_dfs['pop']['co2'], co2_vs_pop_dfs['pop']['pop'], 'CO2 Emissions', 'Population')
scatter_plot (co2_vs_pop_dfs['pop']['co2_log10'], co2_vs_pop_dfs['pop']['pop_log10'], 'CO2 Emissions (log10)', 'Population(log10)')

Plot scatter plot along with linear regression model data and best fit line:

In [ ]:
_regr = linregr_and_plot (co2_vs_pop_dfs['pop']['co2_log10'], co2_vs_pop_dfs['pop']['pop_log10'], 'CO2 Emissions (log10)', 'Population (log10)')

Once plotted on a logarythmic scale, it can be seen that a relationship exists between population and CO2 emissions, however it is not a strong corellation; the 'blob' is widely spaced around the centre line. The R2 score is fairly low (40%), meaning that the model is only a reasonable predictor.

### Birthrate vs Emissions

Does a fast growing population also mean more CO2? Using the linear regression plot function:

In [ ]:
_regr = linregr_and_plot (co2_vs_pop_dfs['br']['co2_log10'],co2_vs_pop_dfs['br']['br_log10'],'CO2 Emissions (KT) (log10)', 'Birthrate (in 1000) (log10)')


### Conclusion


There does not appear to be a definite relationship between birthrate and CO2 emissions. Visually, there is a loose trend that lower birthrate tends to equate with higher climate emissions, and vice versa. Generally, it is understood that poorer countries often have higher birthrates than 'first world' nations, and that the larger developed nations generate the most CO2. 

---
<a id="2_q5_intro"></a>
<font color=#00c382><h2>Q5: Describe the Trends in Rural vs Urban land use by Region.</h2></font>


I have selected the following series to answer this question:

StudyCodes: AG.LND

|Series Code|Description|Unit|
|-|-|-|
AG.LND.TOTL.RU.K2 | Total Rural Land Area | Sq. Km.
AG.LND.TOTL.UR.K2 | Total Urban Land Area | Sq. Km.


In [ ]:
# Extract Study for all countries into a dictionary.
lnd_dfs  = {}
_series_map = {
    'rural':        'AG.LND.TOTL.RU.K2',
    'urban':        'AG.LND.TOTL.UR.K2',
}

series_list = _series_map.keys()

_drop_fields = ['2021']                 
num_fields = [str(x).zfill(2) for x in range(2000,2021)]   

for _series in series_list:
    lnd_dfs  [_series] = country_data_df.loc[country_data_df['SeriesCode'] == _series_map[_series]]
    lnd_dfs  [_series].drop(columns=_drop_fields, axis=1, inplace=True)


We can compare changes in urban and rural land usage, giving a trend. These data series are not directly compariable with the others I have selected, as the only data points are for the years 2000 and 2015.

In [ ]:
# There is only data in column 2000 and 2015
urban_vs_rural_df = pd.merge(lnd_dfs['urban'][['Country','Region','Subregion','2000','2015']], lnd_dfs['rural'][['Subregion','2000','2015']], on='Subregion', suffixes=(['_urban','_rural']), how='outer')

# Calculate the difference in land usage
urban_vs_rural_df['diff_urban'] = urban_vs_rural_df['2015_urban'] - urban_vs_rural_df['2000_urban']
urban_vs_rural_df['diff_rural'] = urban_vs_rural_df['2015_rural'] - urban_vs_rural_df['2000_rural']

urban_vs_rural_df['increased_urban'] = urban_vs_rural_df['diff_urban'] > 0.0
urban_vs_rural_df['decreased_urban'] = urban_vs_rural_df['diff_urban'] < 0.0
urban_vs_rural_df['increased_rural'] = urban_vs_rural_df['diff_rural'] > 0.0
urban_vs_rural_df['decreased_rural'] = urban_vs_rural_df['diff_rural'] < 0.0



In [ ]:
urban_vs_rural_df

These graphs show the distribution of the increase in urban and rural land per region:

In [ ]:
_plot_df = urban_vs_rural_df[['Country', 'Subregion', 'increased_urban', 'increased_rural']]

sns.set_theme()
plt.figure(figsize=(10, 8))
ax = sns.barplot(data=_plot_df, x='increased_urban', hue="Subregion", legend='brief');
plt.xlabel('Distribution', fontsize = 15)
plt.title('Countries that increased Urban land by Subregion', fontsize = 20)
ax.xaxis.tick_top()
plt.legend(loc='center left', bbox_to_anchor=(1.0, 0.5))
plt.rcParams['axes.prop_cycle'] = cycler(color=cmap_terrain_as_list[::32])
plt.show();

plt.figure(figsize=(10, 8))
ax = sns.barplot(data=_plot_df, x='increased_rural', hue="Subregion", legend='brief');
plt.xlabel('Distribution', fontsize = 15)
plt.title('Countries that increased Rural land by Subregion', fontsize = 20)
ax.xaxis.tick_top()
plt.legend(loc='center left', bbox_to_anchor=(1.0, 0.5))
plt.rcParams['axes.prop_cycle'] = cycler(color=cmap_terrain_as_list[::32])
plt.show();


Another fun way to view this data is as follows:

In [ ]:
sns.set_theme()
plt.figure(figsize=(10, 8))
ax = sns.displot(data=_plot_df, x='increased_urban', hue="Subregion", multiple="stack", kind="kde", warn_singular=False);
plt.xlabel('Distribution', fontsize = 12)
plt.title('Countries that increased Urban land by Subregion', fontsize = 14)
plt.rcParams['axes.prop_cycle'] = cycler(color=cmap_terrain_as_list[::32])
plt.show();

sns.set_theme()
plt.figure(figsize=(10, 8))
ax = sns.displot(data=_plot_df, x='increased_rural', hue="Subregion", multiple="stack", kind="kde", warn_singular=False);
plt.xlabel('Distribution', fontsize = 12)
plt.title('Countries that increased Rural land by Subregion', fontsize = 14)
plt.rcParams['axes.prop_cycle'] = cycler(color=cmap_terrain_as_list[::32])
plt.show();

Compare with total as a percentage?

In [ ]:
_df = urban_vs_rural_df[['increased_urban', 'increased_rural', 'decreased_urban', 'decreased_rural']]

In [ ]:
_df['decreased_rural'].value_counts()

In [ ]:
#create pie chart
colors = sns.color_palette('terrain_r')[0:2]
plt.pie(_df['increased_urban'].value_counts(), labels = ['Yes','No'], colors = colors, autopct='%.0f%%')
plt.title('Countries that increased Urban Land Usage')
plt.show()

#create pie chart
colors = sns.color_palette('terrain_r')[0:2]
plt.pie(_df['decreased_urban'].value_counts(), labels = ['No','Yes'], colors = colors, autopct='%.0f%%')
plt.title('Countries that decreased Urban Land Usage')
plt.show()

#create pie chart
colors = sns.color_palette('terrain')[0:2]
plt.pie(_df['increased_rural'].value_counts(), labels = ['No','Yes'], colors = colors, autopct='%.0f%%')
plt.title('Countries that increased Rural Land Usage')
plt.show()

#create pie chart
colors = sns.color_palette('terrain')[0:2]
plt.pie(_df['decreased_rural'].value_counts(), labels = ['Yes','No'], colors = colors, autopct='%.0f%%')
plt.title('Countries that decreased Rural Land Usage')
plt.show()

### Conclusion
There is a general global trend towards an increase in urban land, and a corresponding decrease in rural land. Policy makers may find this information useful when allocating funding towards rural vs urban development. Knowing where people are moving, and how land use is changing is key to protecting valuable and rare environments.

---
<a id="2_q6_intro"></a>
<font color=#00c382><h2>Q6: Does Land Usage Type Corollate to Total CO2 Emissions?</h2></font>


In this question, I will use data on land usage area, and plot against CO2 emissions.

I will plot each type of land usage data against the emissions for each country. I would intuitively expect to see that a predominance of one type of land usage in an area would have a different effect on overall CO2 output, however whether this has an effect from the data remains to be discovered. From this policy decisions can be informed regarding future planning decisions withe a view to Net Zero.

I have selected the following series to answer this question:

StudyCodes: AG.LND, EN.ATM

|Series Code|Description|Unit|
|-|-|-|
AG.LND.TOTL.K2 | Total Land Area | Sq. Km.
AG.LND.TOTL.RU.K2 | Total Rural Land Area | Sq. Km.
AG.LND.TOTL.UR.K2 | Total Urban Land Area | Sq. Km.
AG.LND.AGRI.K2 | Agricultural Land | Sq. Km.
AG.LND.FRST.K2 | Forested Land | Sq. Km.
EN.ATM.CO2E.KT | CO2 Emissions | Tons x 10^3 (KT)


In [ ]:
# Extract Study for all countries into a dictionary.
_source_dfs  = {}
_series_map = {
    'co2':          'EN.ATM.CO2E.KT',
    'total':        'AG.LND.TOTL.K2',
    'rural':        'AG.LND.TOTL.RU.K2',
    'urban':        'AG.LND.TOTL.UR.K2',
    'agricultural': 'AG.LND.AGRI.K2',
    'forest':       'AG.LND.FRST.K2'    
}

series_list = _series_map.keys()

_drop_fields = ['2021']                 
num_fields = [str(x).zfill(2) for x in range(2000,2021)]   

for _series in series_list:
    _source_dfs [_series] = country_data_df.loc[country_data_df['SeriesCode'] == _series_map[_series]]
    _source_dfs [_series].drop(columns=_drop_fields, axis=1, inplace=True)

_source_dfs.keys()

I am precalculating the log10 values and storing in individual dataframes. I did originally place all these in one dataframe as follows, however there are an uneven number of zero values in each dataset - some countries report data for one series, but not another. 

I intend to drop all zero values from each series. Each series starts at 215 rows, but there are many countries with zero values. I also wish to maintain as many values in each list as possible. This requires a separate dataframe to be used for each analysis phase.

In [ ]:

_series_list = list(_source_dfs.keys())
_series_list .remove('co2')

# Extract Co2 values into a long series
_loop_series = _source_dfs['co2'][num_fields]
_co2_series = pd.concat([_loop_series[col] for col in _loop_series])

# Empty dictionary to store the data
lnd_dfs = {}

# Loop through datasets, extracting into one long series
for _key in list(_series_list):

    _loop_series = _source_dfs[_key][num_fields]
    _col_list = ['co2', _key]
    lnd_dfs[_key] = pd.DataFrame(columns = _col_list)

    _df = pd.concat([_loop_series[col] for col in _loop_series])
    lnd_dfs[_key]['co2'] = list ( _co2_series )
    lnd_dfs[_key][_key] = list ( _df )

del _source_dfs

In [ ]:

# Loop through the datasets, remove zero value rows and perform log10
for _key in list(lnd_dfs.keys()):
    lnd_dfs[_key] = lnd_dfs[_key].loc[(lnd_dfs[_key] > 0.0).all(axis=1)]
    log10_to_column (lnd_dfs[_key], _key)
    log10_to_column (lnd_dfs[_key], 'co2')

lnd_dfs['total'].columns

In [ ]:
# Plot as a scatter plot

# First, actual values, then the log10 value per series
for _key in list(lnd_dfs.keys()):
    print (f'Dataframe: {_key}')
    print (lnd_dfs[_key].head())
    
    _x = lnd_dfs[_key]['co2_log10']
    _y = lnd_dfs[_key][_key+'_log10']
    scatter_plot (_x, _y, x_label= 'CO2 Emissions (KT) (log10)', y_label=f'{_key.capitalize()} Land (Sq. Km.) (log10)')

On one plot:

In [ ]:

# Log10 Values
plt.figure(figsize=(10, 8))
for _key in list(lnd_dfs.keys()):
    _x = lnd_dfs[_key]['co2_log10']
    _y = lnd_dfs[_key][_key+'_log10']
    plt.scatter(_x, _y, s=5)

plt.ylabel('Land (Sq. Km.) (log10)')
plt.xlabel('CO2 Emissions (KT) (log10)')
plt.title('Scatter Plot: Land Series Data vs Total CO2 Emissions')
plt.rcParams['axes.prop_cycle'] = cycler(color=cmap_terrain_as_list[::32])
plt.show();

Using the function from the prior question, we can now easily run a linear regression model and plot for each series:

In [ ]:
# First, actual values, then the log10 value per series
for _key in list(lnd_dfs.keys()):
    print ()
    print (f'[Dataframe]: {_key.capitalize()}')
    print ()
    _x = lnd_dfs[_key]['co2_log10']
    _y = lnd_dfs[_key][_key+'_log10']
    _regr = linregr_and_plot (_x, _y, x_label= 'CO2 Emissions (KT) (log10)', y_label=f'{_key.capitalize()} Land (Sq. Km.) (log10)')
    print ()

### Conclusion

The notable R2 values for the results are:

|Series Code|R2
|-|-|
Total Land      | 22%
Urban Land      | 75%
Argicultural Land | 59%
 
This implies that there is a strong relationship between land used for urban purposes and increased CO2 emissions. This would appear to follow 'common sense' - there are more people and more vehicles in urban land.

There is also a good correllation between land used for agriculture and CO2 emissions. Note that this refers to active farm land, rather than rural land - which could include villages and national parks. Again, as beef cattle accounts for a large percentage of greenhouse gas emissions, this is somewhat expected.

Agricultural land has less of a detrimental effect than urban land.

---
<a id="2_q7_intro"></a>
<font color=#00c382><h2>Q7: Show electrical consumption over time, by region.</h2></font>

An analysis of electrical consuption per capita by region.

I have selected the following series to answer this question:

StudyCodes: EN.ATM

|Series Code|Description|Unit|
|-|-|-|
EG.USE.ELEC.KH.PC    | Electric power consumption |(kWh per capita)




In [ ]:
# Extract Study for all countries

power_dfs = {}
_series_map = {
    'usepc':    'EG.USE.ELEC.KH.PC'
}

series_list = _series_map.keys()

_drop_fields = [str(x).zfill(2) for x in range(2015,2022)]
elec_num_fields = [str(x).zfill(2) for x in range(2000,2015)]      # <-- zero data in >2015

for _series in series_list:
    power_dfs[_series] = country_data_df.loc[country_data_df['SeriesCode'] == _series_map[_series]]
    power_dfs[_series].drop(columns=_drop_fields, axis=1, inplace=True)

power_dfs.keys()

In [ ]:
for _series in series_list:
    print (f"Series: {_series}")
    print()
    print(power_dfs[_series].head(3))
    print()

Now, I will group and sum by subregion and plot:


In [ ]:
for _subregion in subregions:
    _df = power_dfs['usepc'].loc[power_dfs['usepc']['Subregion'] == _subregion]
    if _df.shape[0] > 0:
        plot_data_series_years (dataframe=_df, ylabel="", title="Electricity Consumption (kWh per captia) for "+_subregion, start=2000, end=2014, showlegend=True)

In [ ]:
usepc_subregion_gb = power_dfs['usepc'].groupby('Subregion')[elec_num_fields].sum()

fig, ax = plt.subplots(figsize=(15,6))
usepc_subregion_gb.transpose().plot(kind='line', ax=ax, ylabel="kWh", title='Total Power Consumption (kWh per capita), by Subregion');
plt.legend(loc='center left', bbox_to_anchor=(1.0, 0.5))
plt.rcParams['axes.prop_cycle'] = cycler(color=cmap_terrain_as_list[::8])
plt.show()

In [ ]:
sns.set_theme()
plt.figure(figsize=(10, 8))
ax = sns.heatmap(usepc_subregion_gb.transpose(),cmap="ocean");
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='left')
ax.set_yticklabels(elec_num_fields, rotation=0)
plt.xlabel('Subregions', fontsize = 12)
plt.ylabel('Years', fontsize = 12)
plt.title('Heatmap of Total Power Consumption (kWh per capita), by Subregion', fontsize = 18)
ax.xaxis.tick_top() 

### Conclusion

Electricity consumption per capita is increasing globally. Some countries and regions are increasing are at a faster rate than others. 

Nothern Europe, and in particular the Danes use proportionally more electricity than anywhere else on Earth. It can be cold and dark in the Nordic nations throughout winter, but not notably more so in Denmark than elsewhere.

Policy makers can use this information for both ensuring sufficient future supply security, and to focus industrial export activities of suitable products for such a market.

---
<a id="2_q8_intro"></a>
<font color=#00c382><h2>Q8: Is there a correlation between electricity production method and overall CO2 emissions?</h2></font>

I have selected the following series to answer this question:

|Series Code|Description|Unit|
|-|-|-|
EN.ATM.CO2E.KT | CO2 Emissions | Tons x 10^3 (KT)
EG.ELC.COAL.ZS       | Electricity production from coal sources |(% of total)
EG.ELC.HYRO.ZS       | Electricity production from hydroelectric sources|(% of total)   
EG.ELC.NGAS.ZS       | Electricity production from natural gas sources|(% of total)
EG.ELC.NUCL.ZS       | Electricity production from nuclear sources |(% of total)
EG.ELC.PETR.ZS       | Electricity production from oil sources | (% of total)   
EG.ELC.RNWX.ZS       | Electricity production from renewable sources, excluding hydroelectric | (% of total)
EG.FEC.RNEW.ZS       | Renewable energy consumption |(% of total final energy consumption)
EG.IMP.CONS.ZS       | Energy imports, net |(% of energy use) 


In [ ]:
# Extract Study for all countries

power_dfs = {}
_series_map = {
    'co2':      'EN.ATM.CO2E.KT',
    'coal':     'EG.ELC.COAL.ZS',
    'hyro':     'EG.ELC.HYRO.ZS',
    'ngas':     'EG.ELC.NGAS.ZS',
    'nucl':     'EG.ELC.NUCL.ZS',
    'petr':     'EG.ELC.PETR.ZS',
    'rnwx':     'EG.ELC.RNWX.ZS',
    'impt':     'EG.IMP.CONS.ZS',
}

series_list = _series_map.keys()

_drop_fields = [str(x).zfill(2) for x in range(2016,2022)]
elec_num_fields = [str(x).zfill(2) for x in range(2000,2016)]      # <-- zero data in >2015

for _series in series_list:
    power_dfs[_series] = country_data_df.loc[country_data_df['SeriesCode'] == _series_map[_series]]
    power_dfs[_series].drop(columns=_drop_fields, axis=1, inplace=True)

power_dfs.keys()

In [ ]:
power_dfs['coal'].head(3)

I am defining nuclear energy as 'non-renewable':

> Experts still question whether the world should call nuclear power a "renewable" energy resource. Those who want to classify nuclear energy as renewable cite the fact that it has low carbon emission -- just the way renewable sources such as wind and solar do. Non-renewable fuels, such as natural gas and oil, produce byproducts that harm the environment through global warming emissions. Those opposed to calling nuclear power renewable note that nuclear power plants create harmful waste.


Source: https://sciencing.com/nuclear-energy-renewable-nonrenewable-4579290.html

In [ ]:
non_rnwx = ['coal','ngas','nucl','petr']

_combined_df = pd.DataFrame()
_combined_df = power_dfs['coal'].set_index('ISOcode')[elec_num_fields] + power_dfs['ngas'].set_index('ISOcode')[elec_num_fields] + power_dfs['petr'].set_index('ISOcode')[elec_num_fields] + power_dfs['nucl'].set_index('ISOcode')[elec_num_fields]


In [ ]:
# Merge with country and Subregion data and save into dictionary
power_dfs['non_rnwx'] = pd.merge(_combined_df, power_dfs['coal'][['Country', 'Subregion', 'ISOcode']], on='ISOcode', how='outer')
print (power_dfs['non_rnwx'].head(3))

del _combined_df

In [ ]:
_combined_df = pd.DataFrame()
_combined_df = power_dfs['rnwx'].set_index('ISOcode')[elec_num_fields] + power_dfs['hyro'].set_index('ISOcode')[elec_num_fields]
power_dfs['rnwc'] = pd.merge(_combined_df, power_dfs['rnwx'][['Country', 'Subregion', 'ISOcode']], on='ISOcode', how='outer')


In [ ]:

_series_list = list(power_dfs.keys())
_series_list.remove('co2')

# Extract Co2 values into a long series
_loop_series = power_dfs['co2'][elec_num_fields]
_co2_series = pd.concat([_loop_series[col] for col in _loop_series])

# Empty dictionary to store the data
co2_vs_elec_dfs = {}

# Loop through datasets, extracting into one long series
for _key in list(_series_list):
    _loop_series = power_dfs[_key][elec_num_fields]
    _col_list = ['co2', _key]
    co2_vs_elec_dfs[_key] = pd.DataFrame(columns = _col_list)

    _df = pd.concat([_loop_series[col] for col in _loop_series])
    co2_vs_elec_dfs[_key]['co2'] = list ( _co2_series )
    co2_vs_elec_dfs[_key][_key] = list ( _df )

co2_vs_elec_dfs.keys()


In [ ]:
# log10_to_column function defined previously.
# Loop through the datasets, remove zero value rows and perform log10
for _key in list(co2_vs_elec_dfs.keys()):
    co2_vs_elec_dfs[_key] = co2_vs_elec_dfs[_key].loc[(co2_vs_elec_dfs[_key] > 0.0).all(axis=1)]
    log10_to_column (co2_vs_elec_dfs[_key], _key)
    log10_to_column (co2_vs_elec_dfs[_key], 'co2')


Using the function created previously, we can run a linear regression model for each dataset and plot very easily:

In [ ]:

for _key in list(co2_vs_elec_dfs.keys()):
    print ()
    print (f'[Dataframe]: {_key.capitalize()}')
    print ()
    _x = co2_vs_elec_dfs[_key]['co2_log10']
    _y = co2_vs_elec_dfs[_key][_key+'_log10']
    _regr = linregr_and_plot (_x, _y, y_label= 'CO2 Emissions (KT) (log10)', x_label=f'{_key.capitalize()}% of total production')

    print ()

In [ ]:
del power_dfs

### Conclusion

There are no strong methematical relationships between these datasets alone. While producing more power through renewables would intuitively lead to lower carbon emissions, this is not the case. Other factors other than the percentage of renewables produced are in play.

It should also be considered that there are large CO2 emissions involved in creating renewable infrastucture and components. Nothing is carbon free.

One insight from this is that increasing renweable energy sources without decreasing production using other sources will not produce a reduction in carbon emissions. Those wanting to make truthful statements about energy sources in elections should bear this in mind.


---
<a id="3_intro"></a>
<font color=#009afa><h1>Task 3 - Classification/Clustering</h1></font>


> Use appropriate classification/clustering technique to classify/cluster the chosen data based on data analysis requirements to discover meaningful insights.


Let's look at clustering within the CO2 Emissions data. Is there any relationship between cluster membership based on emissions and the subregion it belongs to?

I have also included the birthrate and deathrate series. Are there any natural clusters among countries by these metrics?




In [ ]:
# Extract Study for all countries

_source_dfs = {}
_series_map = {
    'co2':          'EN.ATM.CO2E.KT',
    'birthrate':    'SP.DYN.CBRT.IN',
    'deathrate':    'SP.DYN.CDRT.IN'
}

series_list = _series_map.keys()
# True?
_drop_fields = [str(x).zfill(2) for x in range(2016,2022)]
num_fields = [str(x).zfill(2) for x in range(2000,2016)]      # <-- zero data in >2015

for _series in series_list:
    _source_dfs[_series] = country_data_df.loc[country_data_df['SeriesCode'] == _series_map[_series]]
    _source_dfs[_series].drop(columns=_drop_fields, axis=1, inplace=True)

_source_dfs.keys()

Perform normalisation across all series, save into cluster_dfs dictionary:

In [ ]:
from sklearn import cluster
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize

cluster_dfs = {}
_to_merge = ['ISOcode', 'Country', 'Subregion','Region']

for _series in _source_dfs.keys():
    _norm_data = pd.DataFrame(normalize(_source_dfs[_series][num_fields]))
    _source_dfs [_series].reset_index(inplace=True)
    _new_df = f"{_series}_norm"
    cluster_dfs[_new_df] = pd.concat([_source_dfs[_series][_to_merge],_norm_data], axis=1, join='outer', ignore_index=True)
    cluster_dfs[_new_df].columns = _to_merge + num_fields

In [ ]:
_series = 'birthrate_norm'
cluster_dfs[_series]

Perform KMeans clustering

In [ ]:
# Define a function to run a kmeans cluster against a series

def kmeans_series (dataframe, n_clusters, numfields, groupby='Country', ):

    # Extract numerical fields columns
    _cluster_columns = cluster_dfs[_series][num_fields]

    _kmeans = KMeans(n_clusters=n_clusters, random_state=69)
    _kmeans.fit(_cluster_columns)

    dataframe['cluster_km']=_kmeans.labels_

    _country_cluster_gb = dataframe.groupby('cluster_km')[groupby].unique()

    _display_number = 6

    # Print the cluster information
    for _cluster, _countries in _country_cluster_gb.items():
        print()
        print (f"Cluster {_cluster + 1} contains {len(_countries)} items, including:")
        print (f"\t{', '.join(_countries[0:_display_number])}", end='')
        if len(_countries) > 1:
            print (f"... +{len(_countries)-_display_number}")
        else:
            print()

    _cluster_ids = [int(str(x).zfill(1)) for x in range(1,n_clusters+1)]

    # Calculate the Sum Square of Errors
    _SSE = []
    for _k in _cluster_ids:
        _k_means = KMeans(n_clusters=_k)
        _k_means.fit(dataframe[num_fields])
        _SSE.append(_k_means.inertia_)

    print ()
    print ("Sum Square of Errors:")
    print(_cluster_ids)
    print (_SSE)

    plt.plot(_cluster_ids, _SSE)
    plt.show();        

    return _kmeans

Cycle through all series to perform clustering

In [ ]:
_n_clusters = 6
for _series in cluster_dfs.keys():
    print ()
    print (f"[Series] {_series}")
    _kmeans_br_by_c = kmeans_series (cluster_dfs[_series], _n_clusters, groupby='Country', numfields=num_fields)
    _kmeans_br_by_sr = kmeans_series (cluster_dfs[_series], _n_clusters, groupby='Subregion', numfields=num_fields)

Interpret above, per series:



In [ ]:
def linregr_and_plot (x_series, y_series, x_label, y_label):

    lin_x = pd.DataFrame(x_series)
    lin_y = pd.DataFrame(y_series)

    # Setting random_state tp ensure consistent results across notebook runs.
    x_train, x_test, y_train, y_test = train_test_split(lin_x , lin_y, test_size=0.33, random_state=69)

    regr=linear_model.LinearRegression()
    regr.fit(x_train,y_train)

    test_y_ = regr.predict(x_test)

    # Modified from workshop on Linear Regression
    print('Coefficients: ', regr.coef_)
    print('Intercept: ', regr.intercept_)
    print("Mean absolute error: %.2f" % np.mean(np.absolute(test_y_-y_test)))
    print("Residual sum of squares (MSE): %.2f" % np.mean((test_y_-y_test)**2))
    print("R2-score: %.2f" % r2_score(test_y_,y_test))

    scatter_plot (x_series, y_series, x_label, y_label, lr=regr, x_train=x_train)
    
    return regr

In [ ]:
def cluster_hf_and_plot (dataframe, pivotcol, x_label, y_label, values, title=''):

    _cluster_pt  = pd.pivot_table(dataframe, columns=pivotcol, values=values)
    _cluster_hier_df = _cluster_pt.transpose().copy()
    _cluster_labels=_cluster_pt.keys()

    # Perform hierarchical clustering
    _Z = linkage(_cluster_hier_df, method="ward")

    ## Plot the dendrogram
    plt.figure(figsize=(15, 6))
    dendrogram(_Z, labels=_cluster_labels)
    plt.xlabel(x_label)
    plt.ylabel(y_label)
    plt.title(title)
    plt.show()

    ## Perform K-means clustering
    kmeans = KMeans(n_clusters=_n_clusters, random_state=69)
    kmeans.fit(_cluster_hier_df)

    ## Assign cluster labels to the DataFrame
    _cluster_hier_df['cluster_h'] = kmeans.labels_

    # Print the cluster information
    _cluster_h_sr_gb = _cluster_hier_df.groupby("Subregion")["cluster_h"].unique()
    for _item, clusters in _cluster_h_sr_gb.items():
        print(f'Region: {_item}: Cluster memberships: {clusters}')

    #_n_clusters = 6
    _cluster_ids = [int(str(x).zfill(1)) for x in range(1,7)]

    # Calculate the Sum Square of Errors
    _SSE = []
    for _k in _cluster_ids:
        _k_means = KMeans(n_clusters=_k)
        _k_means.fit(_cluster_hier_df[num_fields])
        _SSE.append(_k_means.inertia_)

    print ()
    print ("Sum Square of Errors:")
    print(_cluster_ids)
    print (_SSE)

    plt.plot(_cluster_ids, _SSE)
    plt.show();     

    print ("Seaborn Clustermap Chart")
    data = pd.pivot_table(_cluster_hier_df, values=num_fields, index='Subregion');
    sns.clustermap(data, cmap='ocean');
    plt.show();

#del _cluster_sr_pt

Now loop through each series and perform hierarchical clustering for each:

In [ ]:
for _series in cluster_dfs.keys():
    print ()
    print (f"[Series] {_series}")
    cluster_hf_and_plot (cluster_dfs[_series], 'Subregion', 'Subregion', 'Distance', 
                         num_fields, f"Hierarchical Clustering Dendrogram of {_series}")


---
<a id="3_intro"></a>
<font color=#009afa><h1>Task 4. Discussion</h1></font>

> *Provide a brief discussion to provide some recommendations how the policies could be regulated to achieve the wellbeing goals based on data analysis results.*

During the analysis of the data, I attempted to provided answers to the following questions:

- Q1: Are CO2 emissions increasing over time?
- Q2: Rank CO2 emitters by region. Who is worst?
- Q3: Do richer nations generate more CO2?
- Q4: Is there a link between population and birth/death rate to CO2 emissions?
- Q5: Describe the Trends in Rural vs Urban land use by Region.
- Q6: Does Land Usage Type Corollate to Total CO2 Emissions?
- Q7: Show electrical consumption over time, by region.
- Q8: Is there a correlation between electricity production method and overall CO2 emissions?

The analysis and ranking of the CO2 data in particular provided insights:

- **Global CO2 emissions are increasing**, despite COP / Kyoto / Paris agreements.

- In this dataset, **global emissions peaked in 2018**. The trend is upwards.

- In that context, 52 countries out of 215 **(24%) decreased their CO2 emissions** between 2000 and 2020, including the UK and, surprisingly, the USA.
- **2020 had a marked impact** on global emissions due to Pandemic lockdowns.
- **There are consistent high ranking Co2 emitters**, although these change when ranked as per capita figures.
- A number of countries have fallen drastically in the emitter rankings. The UK, for example, has dropped from 7th highest polluter to 15th in that period.
- There is a reasonably well defined **relationship between high GDP per capita and high CO2 emissions per capita**.
- When ranked by CO2/pc, the **majority of emitters in the Top 10 are oil producers**.
- There is quite a lot of movement in the GDP/pc ranking table, including some surprises such as Greenland. **The top three 'rich list' leaders are consistent throughout the period.** (Monaco, Lichtenstein and Luxemberg).
- There is a general global trend towards **an increase in urban land**, and a corresponding decrease in rural land.
- 92% of countries increased urban land usage, while only **n%** increased their allocation of rural land.
- There is a **strong relationship between land used for urban purposes and increased CO2 emissions**. This would appear to follow 'common sense' - there are more people and more vehicles in urban land.
- Conversely, **agricultural land has less of a detrimental effect than urban land**.

## Suggestions

The following policy suggestions can be made to policy makers / legislators wishing to reduce their nations climate impact:

- **Be clear and truthful in climate messaging**, even around election time. Renewable energy sources are vitally important, but they are not going to stop climate change, or even reduce emissions without other nationwide policy and individual changes.
- **Reduce urban sprawl where possible**. Enforce stricter urban planning controls. Enforce reuse of brown-field sites. Better still, **encourage redevelopment**  and/or **reuse of existing urban architecture**.
- **Do not embark on large infrastructure projects**, certainly no large concrete White Elephant projects. 
- Legislate for **integrated parkland and open green spaces** within the urban landscape. 
- **Encourage rewilding** initiatives to return land to a rural state.



---

End of Document

---